In [1]:
from datasets import Dataset, load_dataset
import json
#from huggingface_hub import HfApi, login
import pandas as pd
import numpy as np

In [3]:
dataset = load_dataset("thoughtworks/psychometric_personas")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/233k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100 [00:00<?, ? examples/s]

In [67]:
df = dataset['train'].to_pandas()

In [69]:
df.iloc[0]

version                                                                            v0
archetype                                 The Professional (Service-Oriented Officer)
name                                                                  Travis Derizans
age                                                                                62
location                                                               Santa Rosa, CA
appearance_category                                                   Off-Duty/Hybrid
behavior_category                                              Procedural/By-the-Book
memoir                                              Forced Out — Kevin Maxwell (2020)
memoir_narrative                    I faced relentless pressure after my orientati...
archetype_description               Core trait: Emphasizes community service and p...
memoir_summary                      In "Forced Out," Kevin Maxwell recounts his ex...
presenting_problems                 [Difficulty adapti

In [81]:
from typing import Dict, List, Optional

def create_persona_narrative(row: pd.Series, 
                           sections: List[str] = None) -> str:
    """
    Combines selected persona sections into a natural narrative paragraph.
    
    Parameters:
    -----------
    row : pd.Series
        A row from the personas dataframe
    sections : List[str], optional
        List of column names to include. If None, uses default selection.
        
    Returns:
    --------
    str : Combined narrative text
    """
    
    # Default sections if none specified
    if sections is None:
        sections = [
            'archetype_description',
            'appearance', 
            'behavior',
            'mood_affect',
            'speech',
            'emotional_behavioral_functioning',
            'social_functioning'
        ]
    
    # Build the narrative
    narrative_parts = []
    
    # Start with basic intro
    name = row.get('name', 'This person')
    age = row.get('age', '')
    location = row.get('location', '')
    
    intro = f"{name}"
    if age:
        intro += f", {age} years old"
    if location:
        intro += f" from {location},"
    intro += " presents as someone who has these attributes: "
    
    narrative_parts.append(intro)
    
    # Process each section with natural transitions
    transitions = {
        'archetype_description': "",
        'appearance': "Physically, they ",
        'behavior': "In terms of behavior, they ",
        'mood_affect': "Their emotional presentation shows ",
        'speech': "When speaking, they demonstrate ",
        'thought_content': "Their thinking patterns reveal ",
        'emotional_behavioral_functioning': "Emotionally, they ",
        'social_functioning': "Socially, they ",
        'summary_of_psychological_profile': "Overall, "
    }
    
    for i, section in enumerate(sections):
        if section in row and pd.notna(row[section]) and str(row[section]).strip():
            content = str(row[section]).strip()
            
            # Add transition phrase
            if section in transitions:
                transition = transitions[section]
            else:
                transition = "Additionally, they "
            
            # Clean up the content to flow better
            content = content.replace('\n', ' ').replace('  ', ' ')
            
            # Make first letter lowercase if it follows a transition
            if transition and content:
                content = content[0].lower() + content[1:] if len(content) > 1 else content.lower()
            
            # Ensure proper sentence ending
            if content and not content.endswith(('.', '!', '?')):
                content += '.'
            
            narrative_parts.append(f"{transition}{content}")
    
    # Join all parts
    full_narrative = " ".join(narrative_parts)
    
    # Clean up any double spaces or formatting issues
    full_narrative = " ".join(full_narrative.split())
    
    return full_narrative

In [73]:
df['persona_narrative'] = df.apply(create_persona_narrative, axis=1)

In [75]:
df['persona_narrative'][0]

'Travis Derizans, 62 years old from Santa Rosa, CA, presents as someone who has these attributes: Core trait: Emphasizes community service and procedural justice. Focus: De-escalation, fairness, empathy, and following policy. Strengths: Builds trust with citizens and fosters legitimacy; Effective in diverse communities with varied expectations; Provides steady and consistent interactions. Challenges: Can become frustrated in high-conflict or aggressive departments; May struggle with hands-on tactics if they view de-escalation as the only acceptable method. Physically, they wears worn-in plaid flannel over a fitted tee, relaxed jeans with slight fading, and scuffed loafers, subtle tattoos glimpsed on forearms. In terms of behavior, they meticulously reviews department policies daily; follows reporting protocols carefully; uses consistent de-escalation language; visibly listens without interruption; precise note-taking habits. Their emotional presentation shows generally composed with oc

In [53]:
from sentence_transformers import SentenceTransformer

def generate_embeddings(df: pd.DataFrame, 
                       text_column: str = 'persona_narrative',
                       model_name: str = "Qwen/Qwen3-Embedding-0.6B") -> np.ndarray:
    """
    Generates text embeddings for the specified column using sentence transformers.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing the text to embed
    text_column : str
        Name of the column containing text to embed
    model_name : str
        Name of the sentence transformer model to use
        
    Returns:
    --------
    np.ndarray : Array of embeddings with shape (n_samples, embedding_dim)
    """
    
    # Load the model
    model = SentenceTransformer(model_name)
    
    # Get the text data
    texts = df[text_column].fillna('').astype(str).tolist()
    
    # Generate embeddings
    embeddings = model.encode(texts, show_progress_bar=True)
    
    return embeddings

In [54]:
embeddings = generate_embeddings(df, 'persona_narrative')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [57]:
embeddings.shape

(100, 1024)

In [77]:
df['embedding'] = [embeddings[i] for i in range(100)]

In [104]:
def create_embedding_visualization_json(df: pd.DataFrame, 
                                       embedding_column: str = 'embedding',
                                       output_file: Optional[str] = None,
                                       include_columns: Optional[List[str]] = None,
                                       pca_components: int = 2,
                                       umap_components: int = 2,
                                       umap_params: Dict = None) -> Dict:
    """
    Creates a JSON structure with 2D PCA and UMAP transformations of embeddings
    along with all text columns from the dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with embeddings and text columns
    embedding_column : str
        Name of column containing embedding vectors
    output_file : str, optional
        If provided, saves JSON to this file
    include_columns : List[str], optional
        Specific columns to include. If None, includes all text columns
    pca_components : int
        Number of PCA components (default 2)
    umap_components : int  
        Number of UMAP components (default 2)
    umap_params : Dict, optional
        Additional UMAP parameters
        
    Returns:
    --------
    Dict : JSON-ready dictionary with embeddings and metadata
    """
    
    # Default UMAP parameters
    if umap_params is None:
        umap_params = {
            'n_neighbors': 15,
            'min_dist': 0.1,
            'metric': 'cosine',
            'random_state': 42
        }
    
    # Extract embeddings
    embeddings = np.array(df[embedding_column].tolist())
    
    # Standardize embeddings for better PCA results
    scaler = StandardScaler()
    embeddings_scaled = scaler.fit_transform(embeddings)
    
    # PCA transformation
    print("Performing PCA transformation...")
    pca = PCA(n_components=pca_components, random_state=42)
    pca_embeddings = pca.fit_transform(embeddings_scaled)
    
    # UMAP transformation
    print("Performing UMAP transformation...")
    umap_reducer = umap.UMAP(
        n_components=umap_components,
        **umap_params
    )
    umap_embeddings = umap_reducer.fit_transform(embeddings)
    
    # Determine which columns to include
    if include_columns is None:
        # Include all text/string columns and key numeric ones
        text_columns = []
        for col in df.columns:
            if col != embedding_column:  # Skip embedding column
                # Include if it's string type or important numeric columns
                if (df[col].dtype == 'object' or 
                    col in ['age', 'id', 'index'] or
                    df[col].dtype in ['int64', 'float64'] and col in ['age']):
                    text_columns.append(col)
        include_columns = text_columns
    
    # Build the JSON structure
    json_data = {
        "metadata": {
            "total_points": len(df),
            "original_embedding_dim": embeddings.shape[1],
            "pca_explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
            "pca_total_variance_explained": float(pca.explained_variance_ratio_.sum()),
            "umap_parameters": umap_params,
            "included_columns": include_columns
        },
        "embeddings": {
            "pca": [],
            "umap": []
        },
        "data": []
    }
    
    # Process each data point
    print("Building JSON structure...")
    for i in range(len(df)):
        # PCA coordinates
        pca_point = {
            "x": float(pca_embeddings[i, 0]),
            "y": float(pca_embeddings[i, 1])
        }
        if pca_components > 2:
            for j in range(2, pca_components):
                pca_point[f"z{j-1}"] = float(pca_embeddings[i, j])
        
        # UMAP coordinates  
        umap_point = {
            "x": float(umap_embeddings[i, 0]),
            "y": float(umap_embeddings[i, 1])
        }
        if umap_components > 2:
            for j in range(2, umap_components):
                umap_point[f"z{j-1}"] = float(umap_embeddings[i, j])
        
        # Add to embeddings
        json_data["embeddings"]["pca"].append(pca_point)
        json_data["embeddings"]["umap"].append(umap_point)
        
        # Text and metadata for this point
        point_data = {"index": i}
        for col in include_columns:
            if col in df.columns:
                value = df.iloc[i][col]
                # Handle different data types
                if isinstance(value, (int, float, bool)):
                    point_data[col] = value
                elif isinstance(value, (list, dict, np.ndarray)):
                    point_data[col] = value if not isinstance(value, np.ndarray) else value.tolist()
                else:
                    point_data[col] = str(value)
        
        json_data["data"].append(point_data)

    print(json_data.keys())
    
    # Save to file if requested
    if output_file:
        print(f"Saving to {output_file}...")
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Created JSON with {len(df)} points")
    print(f"📊 PCA explained variance: {json_data['metadata']['pca_total_variance_explained']:.3f}")
    print(f"📋 Included {len(include_columns)} columns: {', '.join(include_columns[:5])}{'...' if len(include_columns) > 5 else ''}")
    
    return json_data

In [106]:
visualization_json = create_embedding_visualization_json(
    df,
    output_file='personas.json'
)

Performing PCA transformation...
Performing UMAP transformation...


/opt/anaconda3/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Building JSON structure...
dict_keys(['metadata', 'embeddings', 'data'])
Saving to personas.json...
✅ Created JSON with 100 points
📊 PCA explained variance: 0.189
📋 Included 27 columns: version, archetype, name, age, location...
